# 전체 파일 통합을 위한 노트북

## 0. 라이브러리 & 설정

In [2]:
import pandas as pd
import glob
import os

# ── 파일이 있는 폴더 경로 
DATA_DIR1 = "../../data/raw/channels/csv"

print("DATA_DIR :", os.path.abspath(DATA_DIR1))


DATA_DIR : /Users/sj.kang/Desktop/project2/SKN30-2nd-1Team/data/raw/channels/csv


## 1. channels 파일 전체 통합

`channels_must_*.csv` 패턴으로 파일을 자동 탐색해서 하나로 합칩니다.


In [8]:
# 파일 목록 탐색
ch_files = sorted(glob.glob(os.path.join(DATA_DIR1, "channels_must_*.csv")))

print(f"발견된 channels 파일: {len(ch_files)}개")
for f in ch_files:
    print(f"  {os.path.basename(f)}")


발견된 channels 파일: 164개
  channels_must_0_49.csv
  channels_must_1000_1049.csv
  channels_must_100_149.csv
  channels_must_1050_1099.csv
  channels_must_1100_1149.csv
  channels_must_1150_1199.csv
  channels_must_1200_1249.csv
  channels_must_1250_1299.csv
  channels_must_1300_1349.csv
  channels_must_1350_1399.csv
  channels_must_1400_1449.csv
  channels_must_1450_1499.csv
  channels_must_1500_1549.csv
  channels_must_150_199.csv
  channels_must_1550_1599.csv
  channels_must_1600_1649.csv
  channels_must_1650_1699.csv
  channels_must_1700_1749.csv
  channels_must_1750_1799.csv
  channels_must_1800_1849.csv
  channels_must_1850_1899.csv
  channels_must_1900_1949.csv
  channels_must_1950_1999.csv
  channels_must_2000_2049.csv
  channels_must_200_249.csv
  channels_must_2050_2099.csv
  channels_must_2100_2149.csv
  channels_must_2150_2199.csv
  channels_must_2200_2249.csv
  channels_must_2250_2299.csv
  channels_must_2300_2349.csv
  channels_must_2350_2399.csv
  channels_must_2400_2449.csv

In [9]:
# 파일별로 읽어서 리스트에 쌓기
ch_list = []
for f in ch_files:
    tmp = pd.read_csv(f)
    tmp["source_file"] = os.path.basename(f)   # 어느 파일에서 왔는지 추적용
    ch_list.append(tmp)
    print(f"  {os.path.basename(f):45s} {len(tmp):>4}행")

print()
all_channels = pd.concat(ch_list, ignore_index=True)
print(f"통합 후 shape : {all_channels.shape}")


  channels_must_0_49.csv                          50행
  channels_must_1000_1049.csv                     50행
  channels_must_100_149.csv                       47행
  channels_must_1050_1099.csv                     49행
  channels_must_1100_1149.csv                     50행
  channels_must_1150_1199.csv                     50행
  channels_must_1200_1249.csv                     50행
  channels_must_1250_1299.csv                     50행
  channels_must_1300_1349.csv                     50행
  channels_must_1350_1399.csv                     48행
  channels_must_1400_1449.csv                     46행
  channels_must_1450_1499.csv                     50행
  channels_must_1500_1549.csv                     50행
  channels_must_150_199.csv                       49행
  channels_must_1550_1599.csv                     48행
  channels_must_1600_1649.csv                     49행
  channels_must_1650_1699.csv                     49행
  channels_must_1700_1749.csv                     48행
  channels_must_1750_1799.cs

In [10]:
# 중복 channel_id 확인
dup = all_channels.duplicated(subset="channel_id")
print(f"중복 channel_id : {dup.sum()}개")

if dup.sum() > 0:
    print("중복 행 제거")
    all_channels = all_channels.drop_duplicates(subset="channel_id").reset_index(drop=True)
    print(f"제거 후 shape  : {all_channels.shape}")


중복 channel_id : 0개


In [11]:
# null 현황
print("=== null 현황 ===")
print(all_channels.isnull().sum())
print()
all_channels.head(3)


=== null 현황 ===
channel_id             0
title                  0
published_at           0
country             1043
uploads_playlist       0
subscriber_count       0
view_count             0
video_count            0
source_file            0
dtype: int64



,channel_id,title,published_at,country,uploads_playlist,subscriber_count,view_count,video_count,source_file
0,UC8EiJ1MgI0S0UAqmWaTcoBw,기준TV,2018-12-18T10:07:33Z,NaN,UU8EiJ1MgI0S0UAqmWaTcoBw,3,495,17,channels_must_0_49.csv
1,UCTdCqgNr9RFJWXK8qZgxL5g,James1004,2008-03-17T12:33:16Z,NaN,UUTdCqgNr9RFJWXK8qZgxL5g,14,6,2,channels_must_0_49.csv
2,UCMCcdMTgIzzdpa2iP8RtS8g,ted youngtae Noh,2011-11-11T01:21:17Z,KR,UUMCcdMTgIzzdpa2iP8RtS8g,11,280,4,channels_must_0_49.csv


In [12]:
# 저장
out_ch = os.path.join(DATA_DIR1, "all_channels.csv")
all_channels.to_csv(out_ch, index=False, encoding="utf-8-sig")
print(f"저장 완료 : {out_ch}")
print(f"shape     : {all_channels.shape}")


저장 완료 : ../../data/raw/channels/csv/all_channels.csv
shape     : (8108, 9)


## 2. videos 파일 전체 통합

`videos_*.csv` 패턴으로 파일을 탐색합니다.  
각 파일에 **컬럼 밀림 오염 행**(published_at에 날짜 대신 숫자가 들어간 행)이  
포함되어 있을 수 있으므로 파일마다 필터링합니다.


In [14]:
DATA_DIR2 = "../../data/raw/videos/csv"


In [15]:
vi_files = sorted(glob.glob(os.path.join(DATA_DIR2, "videos_*.csv")))

print(f"발견된 videos 파일: {len(vi_files)}개")
for f in vi_files:
    print(f"  {os.path.basename(f)}")


발견된 videos 파일: 164개
  videos_0000-0049.csv
  videos_0050-0099.csv
  videos_0100-0149.csv
  videos_0150-0199.csv
  videos_0200-0249.csv
  videos_0250-0299.csv
  videos_0300-0349.csv
  videos_0350-0399.csv
  videos_0400-0449.csv
  videos_0450-0499.csv
  videos_0500-0549.csv
  videos_0550-0599.csv
  videos_0600-0649.csv
  videos_0650-0699.csv
  videos_0700-0749.csv
  videos_0750-0799.csv
  videos_0800-0849.csv
  videos_0850-0899.csv
  videos_0900-0949.csv
  videos_0950-0999.csv
  videos_1000-1049.csv
  videos_1050-1099.csv
  videos_1100-1149.csv
  videos_1150-1199.csv
  videos_1200-1249.csv
  videos_1250-1299.csv
  videos_1300-1349.csv
  videos_1350-1399.csv
  videos_1400-1449.csv
  videos_1450-1499.csv
  videos_1500-1549.csv
  videos_1550-1599.csv
  videos_1600-1649.csv
  videos_1650-1699.csv
  videos_1700-1749.csv
  videos_1750-1799.csv
  videos_1800-1849.csv
  videos_1850-1899.csv
  videos_1900-1949.csv
  videos_1950-1999.csv
  videos_2000-2049.csv
  videos_2050-2099.csv
  videos_2100-

In [16]:
vi_list = []
total_bad = 0

for f in vi_files:
    tmp = pd.read_csv(f)

    # 오염 행 제거: published_at이 YYYY- 형식이 아닌 행
    bad_mask = ~tmp["published_at"].astype(str).str.match(r"\d{4}-", na=True)
    bad_count = bad_mask.sum()
    total_bad += bad_count

    tmp = tmp[~bad_mask].reset_index(drop=True)
    tmp["source_file"] = os.path.basename(f)
    vi_list.append(tmp)

    print(f"  {os.path.basename(f):40s} {len(tmp):>5}행  (오염 {bad_count}행 제거)")

print()
all_videos = pd.concat(vi_list, ignore_index=True)
print(f"통합 후 shape    : {all_videos.shape}")
print(f"총 오염 행 제거  : {total_bad}개")


  videos_0000-0049.csv                       654행  (오염 0행 제거)
  videos_0050-0099.csv                      1282행  (오염 0행 제거)
  videos_0100-0149.csv                      1384행  (오염 0행 제거)
  videos_0150-0199.csv                      1586행  (오염 0행 제거)
  videos_0200-0249.csv                      1783행  (오염 0행 제거)
  videos_0250-0299.csv                      2031행  (오염 0행 제거)
  videos_0300-0349.csv                      1979행  (오염 0행 제거)
  videos_0350-0399.csv                      1805행  (오염 0행 제거)
  videos_0400-0449.csv                      1868행  (오염 0행 제거)
  videos_0450-0499.csv                      1755행  (오염 0행 제거)
  videos_0500-0549.csv                      1890행  (오염 0행 제거)
  videos_0550-0599.csv                      2065행  (오염 0행 제거)
  videos_0600-0649.csv                      2084행  (오염 0행 제거)
  videos_0650-0699.csv                      2042행  (오염 0행 제거)
  videos_0700-0749.csv                      2137행  (오염 0행 제거)
  videos_0750-0799.csv                      2351행  (오염 0행 제거)
  videos

In [17]:
# 타입 변환
all_videos["published_at"] = pd.to_datetime(all_videos["published_at"], utc=True, errors="coerce")
all_videos["is_shorts"]    = all_videos["is_shorts"].astype(str).str.lower() == "true"

# published_at / channel_id null 제거
before = len(all_videos)
all_videos = all_videos.dropna(subset=["channel_id", "published_at"]).reset_index(drop=True)
print(f"null 제거 : {before - len(all_videos)}행 → {len(all_videos):,}행 남음")


null 제거 : 0행 → 377,579행 남음


In [18]:
# 중복 video_id 확인
dup_vi = all_videos.duplicated(subset="video_id")
print(f"중복 video_id : {dup_vi.sum()}개")

if dup_vi.sum() > 0:
    all_videos = all_videos.drop_duplicates(subset="video_id").reset_index(drop=True)
    print(f"제거 후 shape : {all_videos.shape}")


중복 video_id : 600개
제거 후 shape : (376979, 12)


In [19]:
# null 현황
print("=== null 현황 ===")
print(all_videos.isnull().sum())
print()
all_videos.head(3)


=== null 현황 ===
idx                  0
channel_id           0
channel_title        0
video_id             0
video_title          1
published_at         0
duration_sec      1343
is_shorts            0
view_count          12
like_count       13383
comment_count     9217
source_file          0
dtype: int64



,idx,channel_id,channel_title,video_id,video_title,published_at,duration_sec,is_shorts,view_count,like_count,comment_count,source_file
0,0,UCo3Yj54VtkEvQX9cLHKklzw,장정숙,eM25H_bOkJ4,171024 [장정숙 의원] 2017 국정감사_국립대 신입간호사 임금착취 문제 해결,2018-05-28 02:16:23+00:00,66.0,False,75.0,0.0,0.0,videos_0000-0049.csv
1,0,UCo3Yj54VtkEvQX9cLHKklzw,장정숙,KoovsJOVOEg,[국회의원 장정숙] 전반기 교육문화체육관광위원회 활동,2018-07-03 04:16:52+00:00,224.0,False,226.0,10.0,0.0,videos_0000-0049.csv
2,0,UCo3Yj54VtkEvQX9cLHKklzw,장정숙,VhXviHrA_yw,"191211 언제나 국민의편, 국회의원 장정숙",2019-12-11 05:58:53+00:00,86.0,False,192.0,2.0,0.0,videos_0000-0049.csv


In [20]:
# 저장
out_vi = os.path.join(DATA_DIR2, "all_videos.csv")
all_videos.to_csv(out_vi, index=False, encoding="utf-8-sig")
print(f"저장 완료 : {out_vi}")
print(f"shape     : {all_videos.shape}")


저장 완료 : ../../data/raw/videos/csv/all_videos.csv
shape     : (376979, 12)
